# Challenge 05 Tuned: Gradient boosting

## Objective

This notebook is intentionally designed for a heavier search budget.
The goal is to push the selected model beyond the previous local baseline using a two-stage search:

1. A broad coarse search across the most promising region.
2. A fine search built automatically around the best coarse configuration.

These notebooks are expected to take substantially longer than the `Ready` versions.
They are prepared for manual execution and are not guaranteed to exceed 0.90, but they are explicitly configured to chase that target as hard as this model family reasonably allows.

## 0. Load libraries

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655
OUTER_TEST_SIZE = 0.20
SEARCH_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 1. Configure the model family and the coarse search space

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

MODEL_NAME = "Gradient boosting"
NOTEBOOK_SLUG = "challenge_05_gradient_boosting_tuned"
BASELINE_REFERENCE = {
    "previous_local_best_validation_accuracy": 0.6710,
    "goal": "Probe whether a slower, deeper boosting configuration can recover a strong nonlinear decision boundary."
}

COARSE_PIPELINE = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
)

COARSE_GRID = {
    "n_estimators": [100, 200, 400, 600],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "max_depth": [2, 3, 4],
    "min_samples_leaf": [1, 2, 4, 8],
    "subsample": [0.5, 0.6, 0.7, 0.8, 1.0],
    "max_features": [None, 0.5],
}

## 2. Load the challenge data

In [ ]:
BASE_DIR = Path.cwd()
TRAIN_PATH = BASE_DIR / "data" / "training.csv"
TEST_PATH = BASE_DIR / "data" / "test.csv"
SAMPLE_PATH = BASE_DIR / "data" / "sample.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

X = train_df.drop(columns=["id", "class"])
y = train_df["class"]
X_test_kaggle = test_df.drop(columns=["id"])

output_dir = BASE_DIR / "output" / NOTEBOOK_SLUG
output_dir.mkdir(parents=True, exist_ok=True)

submission_dir = BASE_DIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

def save_current_figure(filename: str) -> Path:
    path = output_dir / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    return path

def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "confusion_matrix": cm.tolist(),
    }

## 3. Quick audit of the dataset

In [ ]:
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_df.shape)
print("\nClass balance:")
display(train_df["class"].value_counts().sort_index())

print("\nMissing values in training:", int(train_df.isna().sum().sum()))
print("Missing values in test:", int(test_df.isna().sum().sum()))
print("Duplicated training rows:", int(train_df.duplicated().sum()))

## 4. PCA diagnostic

In [ ]:
scaler_for_pca = StandardScaler()
X_scaled_full = scaler_for_pca.fit_transform(X)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled_full)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.80, color="tomato", linestyle="--", label="80% variance")
plt.axhline(0.90, color="darkgreen", linestyle="--", label="90% variance")
plt.title("Cumulative explained variance from PCA")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend()
pca_variance_path = save_current_figure("pca_cumulative_variance.png")
plt.show()

print(f"Saved figure: {pca_variance_path}")
print("Components for 80% variance:", int(np.argmax(cumulative_variance >= 0.80) + 1))
print("Components for 90% variance:", int(np.argmax(cumulative_variance >= 0.90) + 1))

## 5. Create training and validation splits

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=OUTER_TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training split:", X_train.shape, y_train.shape)
print("Validation split:", X_valid.shape, y_valid.shape)

## 6. Coarse search

In [ ]:
coarse_search = GridSearchCV(
    estimator=COARSE_PIPELINE,
    param_grid=COARSE_GRID,
    scoring="accuracy",
    cv=SEARCH_CV,
    n_jobs=1,
    refit=True,
    verbose=2,
)

coarse_search.fit(X_train, y_train)

print("Coarse best CV accuracy:", round(coarse_search.best_score_, 4))
print("Coarse best params:")
print(coarse_search.best_params_)

coarse_results = (
    pd.DataFrame(coarse_search.cv_results_)
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)
display(coarse_results.loc[:, ["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))
coarse_results.to_csv(output_dir / "coarse_cv_results.csv", index=False)

## 7. Build the fine search space around the best coarse configuration

In [ ]:
best_n_estimators = coarse_search.best_params_["n_estimators"]
best_learning_rate = coarse_search.best_params_["learning_rate"]
best_max_depth = coarse_search.best_params_["max_depth"]
best_min_samples_leaf = coarse_search.best_params_["min_samples_leaf"]
best_subsample = coarse_search.best_params_["subsample"]
best_max_features = coarse_search.best_params_["max_features"]

candidate_estimators = sorted(
    {
        value
        for value in [
            max(100, best_n_estimators // 2),
            best_n_estimators,
            best_n_estimators + 100,
            best_n_estimators + 300,
        ]
        if value <= 1000
    }
)

candidate_learning_rates = sorted(
    {
        round(value, 3)
        for value in [
            best_learning_rate / 2,
            best_learning_rate * 0.75,
            best_learning_rate,
            best_learning_rate * 1.25,
            best_learning_rate * 1.5,
        ]
        if 0.005 <= value <= 0.12
    }
)

candidate_subsamples = sorted(
    {
        round(value, 2)
        for value in [
            max(0.4, best_subsample - 0.1),
            best_subsample,
            min(1.0, best_subsample + 0.1),
        ]
    }
)

FINE_PIPELINE = COARSE_PIPELINE
FINE_GRID = {
    "n_estimators": candidate_estimators,
    "learning_rate": candidate_learning_rates,
    "max_depth": sorted({best_max_depth - 1, best_max_depth, best_max_depth + 1} - {0}),
    "min_samples_leaf": sorted({1, best_min_samples_leaf, best_min_samples_leaf + 1, best_min_samples_leaf + 2}),
    "subsample": candidate_subsamples,
    "max_features": list({best_max_features, None, 0.5}),
}

print("Fine grid:")
print(FINE_GRID)

## 8. Fine search

In [ ]:
fine_search = GridSearchCV(
    estimator=FINE_PIPELINE,
    param_grid=FINE_GRID,
    scoring="accuracy",
    cv=SEARCH_CV,
    n_jobs=1,
    refit=True,
    verbose=2,
)

fine_search.fit(X_train, y_train)

print("Fine best CV accuracy:", round(fine_search.best_score_, 4))
print("Fine best params:")
print(fine_search.best_params_)

fine_results = (
    pd.DataFrame(fine_search.cv_results_)
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)
display(fine_results.loc[:, ["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))
fine_results.to_csv(output_dir / "fine_cv_results.csv", index=False)

search = fine_search

## 9. Visualize the fine-search surface

In [ ]:
gb_results = fine_results.copy()
gb_results["n_estimators"] = gb_results["param_n_estimators"].astype(int)
gb_results["learning_rate"] = gb_results["param_learning_rate"].astype(float)
gb_results["max_depth"] = gb_results["param_max_depth"].astype(int)
gb_results["subsample"] = gb_results["param_subsample"].astype(float)

best_depth = gb_results.iloc[0]["max_depth"]
best_subsample = gb_results.iloc[0]["subsample"]

heatmap_data = (
    gb_results[
        (gb_results["max_depth"] == best_depth)
        & (gb_results["subsample"] == best_subsample)
    ]
    .pivot_table(
        index="learning_rate",
        columns="n_estimators",
        values="mean_test_score",
    )
    .sort_index()
)

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="magma")
plt.title(f"Gradient boosting fine-search heatmap\nmax_depth={best_depth}, subsample={best_subsample}")
plt.xlabel("Number of estimators")
plt.ylabel("Learning rate")
heatmap_path = save_current_figure("gb_fine_heatmap.png")
plt.show()

print(f"Saved figure: {heatmap_path}")

importances = pd.Series(search.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
plt.figure(figsize=(12, 8))
sns.barplot(x=importances.values, y=importances.index, palette="rocket")
plt.title("Top gradient boosting feature importances")
plt.xlabel("Importance")
plt.ylabel("Predictor")
importance_path = save_current_figure("gb_feature_importance.png")
plt.show()

print(f"Saved figure: {importance_path}")

## 10. Evaluate the best tuned model on the holdout validation split

In [ ]:
valid_predictions = search.predict(X_valid)
validation_report = evaluate_predictions(y_valid, valid_predictions)
validation_accuracy = validation_report["accuracy"]

print("Validation accuracy:", round(validation_accuracy, 4))
print("Confusion matrix:")
print(np.array(validation_report["confusion_matrix"]))

disp = ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_predictions,
    display_labels=["Undamaged (0)", "Damaged (1)"],
    cmap="Blues",
    colorbar=False,
)
disp.ax_.set_title("Validation confusion matrix")
confusion_path = save_current_figure("validation_confusion_matrix.png")
plt.show()

print(f"Saved figure: {confusion_path}")

## Interpretation

Gradient boosting is being searched here with slower learning rates, deeper weak learners, and broader stochasticity controls.
This is intentionally expensive. If it still lags after this notebook, it is unlikely to be the winning family for this challenge.

## 11. Refit on the full training set and generate the submission file

In [ ]:
final_model = search.best_estimator_
final_model.fit(X, y)
test_predictions = final_model.predict(X_test_kaggle)

submission_df = pd.DataFrame(
    {
        "id": test_df["id"],
        "class": test_predictions.astype(int),
    }
)

submission_path = submission_dir / f"{NOTEBOOK_SLUG}_submission.csv"
submission_df.to_csv(submission_path, index=False)
submission_df.head()

print(f"Submission saved to: {submission_path}")

## 12. Save the tuning summary

In [ ]:
summary_payload = {
    "model_name": MODEL_NAME,
    "strategy": "coarse_to_fine_grid_search",
    "baseline_reference": BASELINE_REFERENCE,
    "coarse_best_cv_accuracy": float(coarse_search.best_score_),
    "coarse_best_params": coarse_search.best_params_,
    "fine_best_cv_accuracy": float(fine_search.best_score_),
    "fine_best_params": fine_search.best_params_,
    "validation_accuracy": validation_accuracy,
    "output_dir": str(output_dir),
    "submission_path": str(submission_path),
}

summary_path = output_dir / "summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2))

print("Summary saved to:", summary_path)
summary_payload